# Does saying “it” make AI think more? — reproduction

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Alestainer/statistics-intuitions/blob/main/notebooks/05-pronouns-thinking-tokens.ipynb)

Loads the exact 16 matched prompt pairs and saved reasoning-token measurements used in the article. No paid request runs automatically.


In [ ]:
from pathlib import Path
import json, urllib.request

RAW = "https://raw.githubusercontent.com/Alestainer/statistics-intuitions/main/"

def load_json(relative_path):
    for path in (Path("../") / relative_path, Path(relative_path)):
        if path.exists():
            return json.loads(path.read_text())
    with urllib.request.urlopen(RAW + relative_path) as response:
        return json.load(response)

benchmark = load_json("data/05-pronouns-thinking-tokens/benchmark.json")
results = load_json("data/05-pronouns-thinking-tokens/results.json")
print(f"{len(benchmark['items'])} prompt pairs; {len(results['per_model'])} models; {results['calls']} calls")


## Inspect one matched pair


In [ ]:
item = benchmark["items"][0]
print("EXPLICIT NAMES\n")
print(item["prompts"]["explicit"])
print("\n\nPRONOUNS / REFERENCES\n")
print(item["prompts"]["referential"])
print("\nExpected:", item["answer"])


## Recompute the primary table


In [ ]:
import pandas as pd

DISPLAY = {
    "openai/gpt-5.6-sol": "GPT-5.6 Sol",
    "google/gemini-3.8-flash": "Gemini 3.8 Flash",
    "anthropic/claude-opus-5": "Claude Opus 5",
    "anthropic/claude-sonnet-5": "Claude Sonnet 5",
    "qwen/qwen3.8-max": "Qwen 3.8 Max",
    "z-ai/glm-5.3-flash": "GLM 5.3 Flash",
}
ORDER = list(DISPLAY)
rows = []
for model in ORDER:
    primary = results["per_model"][model]["primary"]
    accuracy = primary["accuracy"]
    rows.append({
        "model": DISPLAY[model],
        "explicit mean": round(primary["explicit_reasoning_tokens"]["mean"], 1),
        "referential mean": round(primary["referential_reasoning_tokens"]["mean"], 1),
        "mean paired delta": round(primary["paired_delta_referential_minus_explicit"]["mean"], 1),
        "median paired delta": round(primary["paired_delta_referential_minus_explicit"]["median"], 1),
        "explicit correct": f'{round(accuracy["explicit"]["mean"] * 12)}/12',
        "referential correct": f'{round(accuracy["referential"]["mean"] * 12)}/12',
    })
pd.DataFrame(rows)


## Pooled paired result


In [ ]:
pairs = []
for model, model_result in results["per_model"].items():
    for pair in model_result["primary"]["pairs"]:
        pairs.append({"model": DISPLAY[model], **pair})

paired = pd.DataFrame(pairs)
print("Explicit mean:", round(paired["explicit_reasoning_tokens"].mean(), 1))
print("Referential mean:", round(paired["referential_reasoning_tokens"].mean(), 1))
print("Mean paired delta:", round(paired["delta_reasoning_tokens"].mean(), 1))
print("Median paired delta:", round(paired["delta_reasoning_tokens"].median(), 1))
print("Signs:", {
    "increased": int((paired["delta_reasoning_tokens"] > 0).sum()),
    "tied": int((paired["delta_reasoning_tokens"] == 0).sum()),
    "decreased": int((paired["delta_reasoning_tokens"] < 0).sum()),
})
print("Mean input-token delta:", round(paired["delta_prompt_tokens"].mean(), 1))


## Answer quality and ambiguity diagnostic


In [ ]:
def accuracy_totals(result_key):
    explicit = referential = total = 0
    for model_result in results["per_model"].values():
        for pair in model_result[result_key]["pairs"]:
            explicit += int(pair["explicit_correct"])
            referential += int(pair["referential_correct"])
            total += 1
    return explicit, referential, total

primary_accuracy = accuracy_totals("primary")
diagnostic_accuracy = accuracy_totals("ambiguous_diagnostic")
diagnostic_deltas = [
    pair["delta_reasoning_tokens"]
    for model_result in results["per_model"].values()
    for pair in model_result["ambiguous_diagnostic"]["pairs"]
]

print(f"Primary accuracy, explicit / referential: {primary_accuracy[0]}/{primary_accuracy[2]} / {primary_accuracy[1]}/{primary_accuracy[2]}")
print(f"Diagnostic accuracy, explicit / referential: {diagnostic_accuracy[0]}/{diagnostic_accuracy[2]} / {diagnostic_accuracy[1]}/{diagnostic_accuracy[2]}")
print("Diagnostic mean paired delta:", round(pd.Series(diagnostic_deltas).mean(), 1))
print("Diagnostic median paired delta:", round(pd.Series(diagnostic_deltas).median(), 1))


## Plot the model means


In [ ]:
import matplotlib.pyplot as plt

table = pd.DataFrame(rows).set_index("model")
ax = table[["explicit mean", "referential mean"]].plot.barh(figsize=(8, 4.5), color=["#94a3b8", "#7c3aed"])
ax.set_xlabel("Reported reasoning tokens, mean over 12 unambiguous questions")
ax.set_ylabel("")
ax.legend(["Explicit names", "Pronouns / references"])
plt.tight_layout()
plt.show()


## Scope

The four items labelled `diagnostic` are deliberately ambiguous and are excluded from the primary table. Reasoning-token zeroes are retained as reported; missing accounting would remain missing rather than being converted to zero.

The original requests used temperature 0, low reasoning effort and a 2,048-token completion cap. This notebook does not make API calls because models, providers and prices can change.
